In [20]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow multiple libiomp5md
os.environ["OMP_NUM_THREADS"] = "1"           # keep OpenMP under control

In [21]:
path = r"C:\Users\preet\Documents\BRSET\data"
df = pd.read_csv(path + "//labels.csv")

In [22]:
#Extracting required columns

df = df[["patient_id", "image_id", "diabetic_retinopathy", "exam_eye", "focus", "Illuminaton", "image_field", "artifacts"]]

In [23]:
# Creating the quality label for BRSET

cols = ["focus", "Illuminaton", "image_field", "artifacts"]  
df["quality"] = 1 - df[cols].eq(2).any(axis=1).astype(int)

In [24]:
df = (
    df
    .drop(columns=["focus", "Illuminaton", "image_field", "artifacts"])
    .rename(columns={"image_id": "file"})
)

df["file"] = df["file"].astype(str) + ".png"


In [25]:
df = (
    df
    .rename(columns={
        "exam_eye": "laterality",
        "quality": "final_quality",
        "diabetic_retinopathy": "final_icdr"
    })
)

# map laterality: 1 → 0, 2 → 1
df["laterality"] = df["laterality"].map({1: 0, 2: 1})


In [26]:
# keep only patients with exactly 2 images AND same final_icdr across them
ok_patients = df.groupby("patient_id").filter(
    lambda g: (len(g) == 2) and (g["final_icdr"].nunique() == 1)
)["patient_id"].unique()

df = df[df["patient_id"].isin(ok_patients)].reset_index(drop=True)

df = (
    df.groupby("patient_id")
      .filter(lambda g: (len(g) == 2) and (g["final_icdr"].nunique() == 1))
      .reset_index(drop=True)
)

In [27]:
# Create one label per patient
patient_df = (
    df.groupby("patient_id", as_index=False)
      .agg(final_icdr=("final_icdr", "first"))
)

# Separate DR and non-DR patients
dr_patients = patient_df[patient_df["final_icdr"] == 1]
non_dr_patients = patient_df[patient_df["final_icdr"] == 0]

imbalance_ratio = 9

# Downsample non-DR to control class imbalance
non_dr_sampled = non_dr_patients.sample(
    n=imbalance_ratio * len(dr_patients),
    random_state=42
)

# Keep only selected patients
balanced_patient_ids = pd.concat(
    [dr_patients, non_dr_sampled]
)["patient_id"]

# Filter image-level dataframe
df = df[df["patient_id"].isin(balanced_patient_ids)].reset_index(drop=True)

In [28]:
# One label per patient
patient_df = (
    df
    .groupby("patient_id", as_index=False)
    .agg(final_icdr=("final_icdr", "first"))
)

# 70 / 15 / 15 stratified split
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.30,
    stratify=patient_df["final_icdr"],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    stratify=temp_patients["final_icdr"],
    random_state=42
)

# Map patients back to images
train_df = df[df["patient_id"].isin(train_patients["patient_id"])]
val_df   = df[df["patient_id"].isin(val_patients["patient_id"])]
test_df  = df[df["patient_id"].isin(test_patients["patient_id"])]

# Image counts
print("Image counts:")
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

# Patient-level DR ratio
def print_balance(name, patient_ids):
    subset = patient_df[patient_df.patient_id.isin(patient_ids)]
    print(f"{name} DR ratio:", subset.final_icdr.mean())

print_balance("Train", train_patients.patient_id)
print_balance("Val", val_patients.patient_id)
print_balance("Test", test_patients.patient_id)

Image counts:
Train: 6160
Val:   1320
Test:  1320
Train DR ratio: 0.1
Val DR ratio: 0.1
Test DR ratio: 0.1


In [29]:
train_df.head()

,patient_id,file,final_icdr,laterality,final_quality
0,7,img00013.png,0,0,1
1,7,img00014.png,0,1,1
4,11,img00021.png,1,0,1
5,11,img00022.png,1,1,1
6,13,img00025.png,1,0,1


In [30]:
path = r"C:\Users\preet\Documents\BRSET\data"
train_df.to_pickle("brset_train_2826.pkl")
val_df.to_pickle("brset_val_2826.pkl")
test_df.to_pickle("brset_test_2826.pkl")